# Notebook 6: Neural Operators

**UKACM Autumn School: AI for Computational Mechanics**

### Where this picks up

Notebook 5 solved one beam problem. A network $w_\theta(x)$ took a coordinate and returned a
deflection, trained against the residual of

$$EI\,\frac{d^4 w}{dx^4} = q(x), \qquad w(0)=w(L)=0, \qquad w''(0)=w''(L)=0$$

for one particular load $q$. Changing the load meant throwing the weights away and retraining. That
cost was measured at the end of Notebook 5, and it is the reason this notebook exists.

A **neural operator** learns the map between whole functions,

$$G: q \longmapsto w$$

so that one trained model covers a family of load cases. A new load is then a forward pass, not an
optimisation run.

### What you will do

1. Build an exact dataset of load and deflection pairs from the sine series, and verify it against
   the governing equation before using it.
2. Implement and train a **DeepONet**: a branch network reading the load, a trunk network reading
   the query coordinate, combined by an inner product. Then watch that inner product being
   assembled term by term, and measure what the trunk functions actually are.
3. Implement and train a **Fourier Neural Operator** on the same data, and compare the two. Before
   training it, watch what its spectral truncation keeps and what it discards.
4. Time a PINN retrain against one operator forward pass, then break the operator on purpose with a
   load unlike anything it was trained on.
5. Train the FNO on a coarse grid, evaluate it on a fine one, and measure what actually happens.

### Contents

| Part | Topic |
|---|---|
| 1 | Functions in, functions out |
| 2 | DeepONet |
| 3 | Fourier Neural Operator |
| 4 | The comparison, and the honest limitation |
| 5 | Discretisation behaviour |

### On data files

Nothing to download. The training set is generated from a closed-form solution in milliseconds, so
the notebook runs anywhere PyTorch is installed.

In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, SelectionSlider

np.random.seed(0)
torch.manual_seed(0)

# Small 1D tensors on 2 cores: the threading overhead costs more than it saves.
torch.set_num_threads(1)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

print("torch", torch.__version__, "| numpy", np.__version__)
print("threads:", torch.get_num_threads())


---

# Part 1: Functions in, functions out

### What changed

Notebooks 1 to 4 learned a map from a microstructure to a number: an image or a descriptor vector
in, a stiffness out. The output was a scalar.

Notebook 5 learned a map from a coordinate to a value, $x \mapsto w(x)$, with the load fixed. The
input was a scalar.

An operator learns a map from a function to a function. The input is the entire load $q$, the output
is the entire deflection $w$, and the load is now an argument rather than something baked into the
loss. In practice both functions are handled by their values on a grid, but the object being learned
is the map between them.

### The object being learned

For the simply supported beam, the solution operator is the map that takes a load and returns the
deflection it produces:

$$\boxed{\;G: q \longmapsto w, \qquad
G(q) = w \ \ \text{where} \ \
EI\,\frac{d^4 w}{dx^4} = q, \quad w(0)=w(L)=0, \quad w''(0)=w''(L)=0\;}$$

$q$ is a function on $[0, L]$ with units N/m, $w$ is a function on $[0, L]$ with units m, and
$G$ maps one function space to another. It exists and is unique for this problem because
the boundary value problem is well posed. What is being approximated is $G$ itself, written
$G_\theta$ once it has weights in it.

Put the three notebooks side by side:

| | Input | Output | Trained object |
|---|---|---|---|
| Notebooks 1 to 4 | a microstructure image $I$, 64 by 64 | a number, $E$ in GPa | a regression function |
| Notebook 5 | a coordinate $x$ in m | a number, $w(x)$ in m | the solution for **one** load |
| Notebook 6 | a function $q$ on $[0,L]$ | a function $w$ on $[0,L]$ | the solution operator $G$ |

The first row maps a field to a scalar. The second maps a scalar to a scalar, and the load never
appears as an input at all: it is written into the loss, which is why changing it meant retraining.
The third takes the load as an argument, so changing it is a forward pass.

### How accuracy is reported

The same measure as Notebook 5, so the two are directly comparable. For a predicted deflection
$\hat{w}$ and the exact deflection $w_{\text{exact}}$, evaluated on a grid of $N_p$ points $x_j$,

$$\boxed{\;e_{\text{rel}} \;=\;
\frac{\Bigl(\sum_{j=1}^{N_p}\bigl(\hat{w}(x_j) - w_{\text{exact}}(x_j)\bigr)^2\Bigr)^{1/2}}
     {\Bigl(\sum_{j=1}^{N_p} w_{\text{exact}}(x_j)^2\Bigr)^{1/2}}\;}$$

It is dimensionless, and $10^{-3}$ means the prediction is wrong by about 0.1 percent of the size of
the solution. One difference from Notebook 5: there was one solution to score, here there are
hundreds, so $e_{\text{rel}}$ is computed per load case and then averaged over the test set.
Averaging the ratios rather than taking the ratio of the sums stops the few largest deflections
dominating the reported number.

In [ ]:
# --- What each notebook learned, as a picture --------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

# NB1-4: field in, number out
ax = axes[0]
rng = np.random.default_rng(3)
img = (rng.random((16, 16)) > 0.7).astype(float)
ax.imshow(img, cmap="gray", extent=[0, 1, 0, 1], interpolation="nearest")
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.annotate("", xy=(1.42, 0.5), xytext=(1.05, 0.5), xycoords="axes fraction",
            textcoords="axes fraction", annotation_clip=False,
            arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6))
ax.text(1.24, 0.60, "model", ha="center", fontsize=9, transform=ax.transAxes)
ax.text(1.62, 0.5, "$E$", ha="center", va="center", fontsize=15,
        transform=ax.transAxes, color=C_FIT)
ax.set_title("Notebooks 1 to 4\nfield in, number out", fontsize=10)

# NB5: coordinate in, value out
ax = axes[1]
xs = np.linspace(0, 1, 200)
wq = xs*(1 - 2*xs**2 + xs**3)/24
ax.plot(xs, wq, color=C_ALT, lw=2.5)
ax.plot([0.35], [np.interp(0.35, xs, wq)], "o", color=C_FIT, ms=9)
ax.vlines(0.35, 0, np.interp(0.35, xs, wq), color=C_FIT, lw=1, ls=":")
ax.invert_yaxis()
ax.set_xlabel("$x$"); ax.set_ylabel("$w(x)$")
ax.set_title("Notebook 5\n$x \\mapsto w(x)$, one fixed load", fontsize=10)

# NB6: function in, function out
ax = axes[2]
for k, col in zip([1, 2, 3], [C_DATA, C_FIT, C_BAD]):
    ax.plot(xs, np.sin(k*np.pi*xs), color=col, lw=1.6, alpha=0.85)
    # deflections, all scaled by the same factor so the 1/(n pi)^4 decay is visible
    ax.plot(xs, 120*np.sin(k*np.pi*xs)/(k*np.pi)**4, color=col, lw=1.6, ls="--")
ax.set_xlabel("$x$")
ax.set_title("Notebook 6\n$q \\mapsto w$, a family of loads\n"
             "(solid: loads, dashed: deflections, one common scale)", fontsize=10)
plt.tight_layout(); plt.show()


**What the three panels show.** The same table as above, drawn. Left: Notebooks 1 to 4, a
microstructure in and a single stiffness out. Middle: Notebook 5, a coordinate in and the deflection
at that coordinate out, for one fixed load. Right: this notebook, where the solid curves are three
different loads and the dashed curves are the deflections they produce, and a single trained model
has to cover all of them.

Notice in the right panel how much smoother the dashed curves are than the solid ones, and how much
the higher modes shrink. That is the beam integrating four times, and it is the property that makes
$G$ learnable at all.

### Building an exact dataset

**Why a sine mode is exact for this beam.** Take $\phi_n(x) = \sin(n\pi x/L)$. Two facts, and
together they are the whole construction.

First, $\phi_n$ satisfies all four boundary conditions on its own:

$$\phi_n(0) = \phi_n(L) = 0, \qquad
\phi_n''(x) = -\left(\frac{n\pi}{L}\right)^{\!2}\phi_n(x)
\ \Rightarrow\ \phi_n''(0) = \phi_n''(L) = 0$$

so it is zero at both supports and carries no moment there, which is exactly simple support. Any
combination of these modes inherits that, so the boundary conditions never have to be imposed
separately.

Second, $\phi_n$ is an eigenfunction of the operator on the left of the governing equation:

$$\frac{d^4 \phi_n}{dx^4} = \left(\frac{n\pi}{L}\right)^{\!4}\phi_n$$

So write the load as a truncated sine series,

$$q(x) = \sum_{n=1}^{N_m} a_n \sin\!\left(\frac{n\pi x}{L}\right), \qquad a_n \ \text{in N/m}$$

and the fourth-order equation becomes one scalar division per mode. The deflection is

$$\boxed{\;w(x) = \frac{1}{EI}\sum_{n=1}^{N_m} a_n
\left(\frac{L}{n\pi}\right)^{\!4}\sin\!\left(\frac{n\pi x}{L}\right)\;}$$

Differentiating term by term gives $EI\,w'''' = q$ identically, with no discretisation error
anywhere. Sampling random coefficients $a_n$ therefore produces exact training pairs
$(q, G(q))$ at essentially zero cost.

The $(n\pi/L)^{-4}$ factor is worth noticing now, because it explains most of what follows. The beam
is a strong low-pass filter: mode 8 is attenuated by $8^4 = 4096$ relative to mode 1. Fine detail in
the load barely shows up in the deflection.

The next cell checks the formula numerically before anything is built on it.

In [ ]:
# --- Problem constants, and the sine-series solution -------------------------
L, EI = 1.0, 1.0
N_MODES = 8                       # modes present in the training distribution

def q_from_coeffs(a, x):
    '''q(x) = sum_n a_n sin(n pi x / L).  a: (..., N), x: (M,)  ->  (..., M)'''
    a = np.atleast_2d(a)
    nn_ = np.arange(1, a.shape[-1] + 1)[:, None]
    return a @ np.sin(nn_ * np.pi * x[None, :] / L)

def w_from_coeffs(a, x):
    '''Exact deflection for that load, term by term.'''
    a = np.atleast_2d(a)
    nn_ = np.arange(1, a.shape[-1] + 1)[:, None]
    fac = (L / (np.arange(1, a.shape[-1] + 1) * np.pi)) ** 4 / EI
    return (a * fac) @ np.sin(nn_ * np.pi * x[None, :] / L)

print("mode n :  attenuation factor (L/(n pi))^4 / EI")
for n_ in [1, 2, 4, 8, 16]:
    print(f"   {n_:2d}   :  {(L/(n_*np.pi))**4/EI:.3e}")


In [ ]:
# --- Verification: does the formula actually satisfy the equation? -----------
# One random load, checked three ways. Nothing downstream is used until this passes.

rng_chk = np.random.default_rng(11)
a_chk = rng_chk.normal(size=N_MODES) / np.arange(1, N_MODES + 1)

n_fine = 801
x_fine = np.linspace(0, L, n_fine)
h = x_fine[1] - x_fine[0]
q_chk = q_from_coeffs(a_chk, x_fine).ravel()
w_chk = w_from_coeffs(a_chk, x_fine).ravel()

# 1. fourth derivative by the 5-point central difference, over a grid refinement, so the
#    observed convergence rate tells us what kind of error this is.
def d4_residual(nx):
    xf = np.linspace(0, L, nx)
    hf = xf[1] - xf[0]
    qf = q_from_coeffs(a_chk, xf).ravel()
    wf = w_from_coeffs(a_chk, xf).ravel()
    d4f = (wf[:-4] - 4*wf[1:-3] + 6*wf[2:-2] - 4*wf[3:-1] + wf[4:]) / hf**4
    return np.abs(EI * d4f - qf[2:-2]).max() / np.abs(qf).max()

print("1. residual  EI w'''' - q   (5-point central difference)")
print(f"{'nodes':>8s} {'rel max |residual|':>20s} {'ratio to previous':>19s}")
prev = None
for nx in [201, 401, 801, 1601, 3201]:
    e = d4_residual(nx)
    print(f"{nx:>8d} {e:>20.3e} {'' if prev is None else format(prev/e, '19.2f')}")
    prev = e
print("     Read the ratio column. While it sits near 4, the residual is halving in h^2:")
print("     that is the second-order truncation error of the 5-point stencil, not a")
print("     modelling error and not the roundoff floor. The ratio then collapses, because")
print("     a fourth difference divides by h^4 and rounding is amplified by h^-4, so past")
print("     about 1600 nodes refining the grid makes the residual worse. The 801-node run")
print("     is on the truncation side of that turning point.")

# 2. boundary conditions, from the series for w'' rather than from a difference
nn_chk = np.arange(1, N_MODES + 1)
w_chk = w_from_coeffs(a_chk, x_fine).ravel()
w2 = -(a_chk * (L/(nn_chk*np.pi))**2 / EI) @ np.sin(nn_chk[:, None]*np.pi*x_fine[None, :]/L)
print("2. boundary conditions, satisfied term by term rather than enforced")
print(f"     w(0)   = {w_chk[0]:.3e}   w(L)   = {w_chk[-1]:.3e}")
print(f"     w''(0) = {w2[0]:.3e}   w''(L) = {w2[-1]:.3e}")
print("     Every term is sin(n pi x / L), which vanishes at both ends, and so does its")
print("     second derivative, so these four numbers can only ever be zero to roundoff.")
print("     That is a statement about how the basis is built, not an independent check of")
print("     it. Check 3 is the independent one.")

# 3. independent solve of the same BVP, splitting into two Poisson problems
#    u'' = q with u(0)=u(L)=0  (u = EI w''), then w'' = u/EI with w(0)=w(L)=0
n_bvp = 801
x_bvp = np.linspace(0, L, n_bvp); hb = x_bvp[1] - x_bvp[0]
A = (np.diag(-2*np.ones(n_bvp-2)) + np.diag(np.ones(n_bvp-3), 1)
     + np.diag(np.ones(n_bvp-3), -1)) / hb**2
q_b = q_from_coeffs(a_chk, x_bvp).ravel()
u_b = np.zeros(n_bvp); u_b[1:-1] = np.linalg.solve(A, q_b[1:-1])
w_b = np.zeros(n_bvp); w_b[1:-1] = np.linalg.solve(A, u_b[1:-1] / EI)
w_ref = w_from_coeffs(a_chk, x_bvp).ravel()
print(f"3. finite difference BVP solve vs the series, {n_bvp} nodes")
print(f"     relative L2 difference = "
      f"{np.linalg.norm(w_b - w_ref)/np.linalg.norm(w_ref):.3e}   (second-order scheme)")


Checks 1 and 3 are independent of the series and both agree with it at the level expected
from a second-order scheme. Check 2 is a property of the basis rather than a test of it. Taken
together, the series solution is safe to use as ground truth.

### The training distribution, stated explicitly

Coefficients are drawn as

$$a_n = \frac{z_n}{n}, \qquad z_n \sim N(0,1), \qquad n = 1 \ldots 8$$

so that higher modes are present but weaker. Write this down, because Part 4 goes outside it on
purpose and the operator's behaviour there is the honest half of the story.

Loads and deflections are both sampled on a grid of 64 points. Inputs and outputs are normalised by
a single global scale each, computed from the training set only.

In [ ]:
# --- Generate the dataset ----------------------------------------------------
N_TRAIN, N_TEST = 2000, 400
N_GRID = 64

x_grid = np.linspace(0, L, N_GRID)

def sample_coeffs(n_samples, rng, n_modes=N_MODES):
    decay = 1.0 / np.arange(1, n_modes + 1)
    return rng.normal(size=(n_samples, n_modes)) * decay

t0 = time.time()
rng = np.random.default_rng(0)
A_tr = sample_coeffs(N_TRAIN, rng)
A_te = sample_coeffs(N_TEST,  rng)

Q_tr, W_tr = q_from_coeffs(A_tr, x_grid), w_from_coeffs(A_tr, x_grid)
Q_te, W_te = q_from_coeffs(A_te, x_grid), w_from_coeffs(A_te, x_grid)
t_gen = time.time() - t0

Q_SCALE = Q_tr.std()
W_SCALE = W_tr.std()

print(f"generated {N_TRAIN} training and {N_TEST} test pairs on {N_GRID} points "
      f"in {t_gen*1000:.0f} ms")
print(f"  q: shape {Q_tr.shape}, std {Q_SCALE:.4f}")
print(f"  w: shape {W_tr.shape}, std {W_SCALE:.6f}  (four orders of magnitude smaller)")
print(f"  a full finite element campaign of {N_TRAIN} solves would not be free")


In [ ]:
# --- What the family looks like ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for i in range(8):
    axes[0].plot(x_grid, Q_tr[i], lw=1.4, alpha=0.8)
    axes[1].plot(x_grid, W_tr[i], lw=1.4, alpha=0.8)
axes[0].set_xlabel("$x$ (m)"); axes[0].set_ylabel("$q$ (N/m)")
axes[0].set_title("eight sampled loads", fontsize=10)
axes[1].set_xlabel("$x$ (m)"); axes[1].set_ylabel("$w$ (m)")
axes[1].invert_yaxis()
axes[1].set_title("their exact deflections", fontsize=10)
plt.tight_layout(); plt.show()

print("The deflections are far smoother than the loads. The (n pi/L)^-4 factor")
print("removes the high modes, which is why this operator is learnable at all.")


**What the two panels show.** Left: eight loads drawn from the training distribution, each a
random combination of the first eight sine modes. Right: the exact deflection of each, drawn with
the vertical axis inverted so downwards is down.

The loads wiggle and the deflections do not. Every load here contains mode 8, and mode 8 reaches the
deflection divided by $8^4 = 4096$. The operator's job is therefore much easier than it looks: most
of the detail in the left panel is irrelevant to the right one.

---

# Part 2: DeepONet

### The architecture

DeepONet comes from the universal approximation theorem for operators, which says an operator can be
approximated by a sum of products of two families of functions: one depending on the input function,
one on the output coordinate. The network follows that structure directly.

- The **branch** network reads the load sampled at $m$ fixed **sensor locations**
  $x_1 \ldots x_m$, that is the vector $\bigl(q(x_1), \ldots, q(x_m)\bigr)$, and returns $p$
  coefficients $b_k$.
- The **trunk** network reads a single **query coordinate** $y \in [0, L]$ and returns $p$ basis
  values $t_k(y)$.
- The prediction is their inner product over the latent index $k$:

$$\boxed{\;G_\theta(q)(y) \;=\;
\sum_{k=1}^{p} b_k\bigl(q(x_1), \ldots, q(x_m)\bigr)\; t_k(y) \;+\; b_0\;}$$

$m = 64$ sensors, $p = 48$ latent terms, $b_0$ a single learned bias in m. The sum is over $k$, the
latent index, and nothing else couples the two networks: the branch never sees $y$ and the trunk
never sees $q$.

The trunk learns a basis for the space of solutions, the branch learns the coefficients of the
particular solution in that basis. A useful way to read it: the trunk is doing the job of shape
functions, and the branch is doing the job of the solve.

The sensors are fixed. A different sensor layout means a different branch network, which is the main
practical constraint of the architecture.

In [ ]:
# --- Schematic: the DeepONet ------------------------------------------------
fig, ax = plt.subplots(figsize=(11.5, 4.6))
ax.set_xlim(0, 11.5); ax.set_ylim(0, 4.7); ax.axis("off"); ax.grid(False)

def box(x, y, w_, h_, label, col, fs=9.5):
    ax.add_patch(plt.Rectangle((x, y), w_, h_, fc="w", ec=col, lw=1.8,
                               zorder=3, joinstyle="round"))
    ax.text(x + w_/2, y + h_/2, label, ha="center", va="center",
            fontsize=fs, zorder=4)

def arrow(x0, y0, x1, y1, col="k"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=col, lw=1.5))

# --- branch path, top --------------------------------------------------------
xs_s = np.linspace(0, 1, 200)
qs_s = 1.05*np.sin(np.pi*xs_s) + 0.45*np.sin(3*np.pi*xs_s)
ax.plot(0.35 + 1.5*xs_s, 3.55 + 0.42*qs_s, color=C_DATA, lw=1.8, zorder=3)
x_sen = np.linspace(0, 1, 9)
ax.plot(0.35 + 1.5*x_sen, 3.55 + 0.42*(1.05*np.sin(np.pi*x_sen)
        + 0.45*np.sin(3*np.pi*x_sen)), "o", color=C_DATA, ms=4.5, zorder=4)
ax.text(1.10, 4.45, "load sampled at sensors", ha="center", fontsize=9, color=C_DATA)
ax.text(1.10, 2.98, r"$\left(q(x_1),\,\ldots,\,q(x_m)\right)$", ha="center",
        fontsize=10.5, color=C_DATA)
arrow(2.05, 3.75, 3.05, 3.75, C_DATA)
box(3.05, 3.05, 2.2, 1.40, "branch net\n" + r"$m \to 64 \to 64 \to p$", C_DATA)
arrow(5.25, 3.75, 6.52, 2.70, C_DATA)
ax.text(5.80, 3.80, r"$b_k$", ha="center", fontsize=11.5, color=C_DATA)

# --- trunk path, bottom ------------------------------------------------------
ax.plot([0.35, 1.85], [0.95, 0.95], color="k", lw=1.6, zorder=3)
ax.plot([1.30], [0.95], "o", color=C_FIT, ms=8, zorder=4)
ax.text(0.35, 0.60, "$0$", ha="center", fontsize=9)
ax.text(1.85, 0.60, "$L$", ha="center", fontsize=9)
ax.text(1.30, 1.28, "$y$", ha="center", fontsize=11.5, color=C_FIT)
ax.text(1.10, 0.18, "one query coordinate", ha="center", fontsize=9, color=C_FIT)
arrow(2.05, 0.95, 3.05, 0.95, C_FIT)
box(3.05, 0.25, 2.2, 1.40, "trunk net\n" + r"$1 \to 64 \to 64 \to p$", C_FIT)
arrow(5.25, 0.95, 6.52, 2.00, C_FIT)
ax.text(5.80, 0.55, r"$t_k(y)$", ha="center", fontsize=11.5, color=C_FIT)

# --- where they meet ---------------------------------------------------------
ax.add_patch(plt.Circle((6.85, 2.35), 0.50, fc="w", ec="k", lw=1.8, zorder=3))
ax.text(6.85, 2.35, r"$\sum_k$", ha="center", va="center", fontsize=13, zorder=4)
ax.text(7.55, 1.30, "inner product\nover the latent index", ha="center", va="top",
        fontsize=8.5)
arrow(7.40, 2.35, 8.40, 2.35)
ax.text(9.95, 2.35,
        r"$G_\theta(q)(y)=\sum_{k=1}^{p} b_k\,t_k(y)+b_0$",
        ha="center", va="center", fontsize=11,
        bbox=dict(fc="w", ec="0.4", boxstyle="round,pad=0.45"))
ax.text(9.95, 1.62, "the deflection at $y$,  in m", ha="center", fontsize=9)
ax.set_title("DeepONet: the branch reads the load, the trunk reads the query point",
             fontsize=11)
plt.tight_layout(); plt.show()


**What the schematic shows.** The load enters at the top left, sampled at the $m$ fixed sensor
locations marked on it, and the branch turns those $m$ numbers into $p$ coefficients $b_k$. The query
coordinate $y$ enters at the bottom left on its own, and the trunk turns it into $p$ basis values
$t_k(y)$. The two meet only at the circle, where they are combined by the sum over $k$ in the boxed
equation above.

The asymmetry in the picture is the architecture's main property and its main limitation. The trunk
side is a single coordinate, so the output can be asked for anywhere. The branch side is a vector of
fixed length, so the input sampling is baked into the weights.

In [ ]:
# --- DeepONet ----------------------------------------------------------------
def mlp(sizes, act=nn.Tanh):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return nn.Sequential(*layers)

class DeepONet(nn.Module):
    def __init__(self, m=N_GRID, p=48, width=64):
        super().__init__()
        self.branch = mlp([m, width, width, p])       # q at m sensors -> p coefficients
        self.trunk  = mlp([1, width, width, p])       # y              -> p basis values
        self.b0     = nn.Parameter(torch.zeros(1))

    def forward(self, q_sensors, y):
        '''q_sensors: (B, m) normalised loads.  y: (M, 1) query points.
           returns (B, M) normalised deflections.'''
        b = self.branch(q_sensors)                    # (B, p)
        t = torch.tanh(self.trunk(y))                 # (M, p)
        return b @ t.T + self.b0

torch.manual_seed(0)
don = DeepONet()
print("branch:", [64, 64, 64, 48], " trunk:", [1, 64, 64, 48])
print(f"parameters: {sum(p.numel() for p in don.parameters())}")


### Training

Plain supervised learning. Inputs are the normalised load vectors, targets the normalised
deflections on the same grid, loss is the mean squared error. No physics in the loss at all: the
physics is in the dataset, which came from the exact solution.

Note what this buys and what it costs. The operator needs a set of solved problems up front, which
the PINN in Notebook 5 did not. Here those solutions were free. In a real campaign they would come
from finite element runs, and that cost belongs in any honest comparison.

In [ ]:
# --- Tensors -----------------------------------------------------------------
Qtr_t = torch.tensor(Q_tr / Q_SCALE, dtype=torch.float32)
Wtr_t = torch.tensor(W_tr / W_SCALE, dtype=torch.float32)
Qte_t = torch.tensor(Q_te / Q_SCALE, dtype=torch.float32)
Wte_t = torch.tensor(W_te / W_SCALE, dtype=torch.float32)
y_t   = torch.tensor(x_grid.reshape(-1, 1), dtype=torch.float32)

def rel_l2_rows(pred, true):
    '''Per-sample relative L2 error, then the mean. pred, true: (B, M) numpy.'''
    num = np.linalg.norm(pred - true, axis=1)
    den = np.linalg.norm(true, axis=1)
    return float(np.mean(num / den))

print(f"train {tuple(Qtr_t.shape)} -> {tuple(Wtr_t.shape)}")
print(f"test  {tuple(Qte_t.shape)} -> {tuple(Wte_t.shape)}")


In [ ]:
# --- Train the DeepONet ------------------------------------------------------
EPOCHS_DON, BATCH = 200, 64

opt = torch.optim.Adam(don.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=70, gamma=0.3)
lossf = nn.MSELoss()

hist_don = {"train": [], "test": []}
t0 = time.time()
for ep in range(EPOCHS_DON):
    perm = torch.randperm(N_TRAIN)
    running = 0.0
    for i in range(0, N_TRAIN, BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad()
        loss = lossf(don(Qtr_t[idx], y_t), Wtr_t[idx])
        loss.backward(); opt.step()
        running += loss.item() * len(idx)
    sched.step()
    with torch.no_grad():
        te = lossf(don(Qte_t, y_t), Wte_t).item()
    hist_don["train"].append(running / N_TRAIN); hist_don["test"].append(te)
    if (ep + 1) % 50 == 0:
        print(f"  epoch {ep+1:4d}   train MSE {running/N_TRAIN:.3e}   "
              f"test MSE {te:.3e}", flush=True)
t_don = time.time() - t0

with torch.no_grad():
    P_don = don(Qte_t, y_t).numpy() * W_SCALE
err_don = rel_l2_rows(P_don, W_te)
print(f"\nDeepONet trained in {t_don:.1f} s")
print(f"mean relative L2 error on {N_TEST} unseen loads: {err_don:.4e}")


In [ ]:
# --- Predictions on loads the model has never seen ---------------------------
order = np.argsort([np.linalg.norm(P_don[i]-W_te[i])/np.linalg.norm(W_te[i])
                    for i in range(N_TEST)])
picks = [order[0], order[len(order)//2], order[-1]]      # best, median, worst
labels = ["best of 400", "median", "worst of 400"]

fig, axes = plt.subplots(2, 3, figsize=(13, 5.6))
for j, (k, lab) in enumerate(zip(picks, labels)):
    axes[0, j].plot(x_grid, Q_te[k], color=C_DATA, lw=1.8)
    axes[0, j].set_title(f"{lab}: load", fontsize=10)
    axes[0, j].set_ylabel("$q$ (N/m)" if j == 0 else "")
    axes[1, j].plot(x_grid, W_te[k], color=C_ALT, lw=3, alpha=0.6, label="exact")
    axes[1, j].plot(x_grid, P_don[k], color=C_BAD, lw=1.6, ls="--", label="DeepONet")
    e = np.linalg.norm(P_don[k]-W_te[k])/np.linalg.norm(W_te[k])
    axes[1, j].set_title(f"relative $L_2$ = {e:.2e}", fontsize=10)
    axes[1, j].invert_yaxis(); axes[1, j].set_xlabel("$x$ (m)")
    axes[1, j].set_ylabel("$w$ (m)" if j == 0 else "")
axes[1, 0].legend(fontsize=8, loc="lower center")
plt.tight_layout(); plt.show()


**What the six panels show.** The test set ranked by relative $L_2$ error, with the best case
on the left, the median in the middle and the worst on the right. Top row is the load, bottom row is
the deflection, green for exact and red dashed for the DeepONet, with the error of that individual
case in each title.

The column to judge the model on is the right-hand one, because that is the case a user would
actually be unlucky enough to meet. Compare its error with the mean printed above: the worst of 400
is worse than the average, but it is still the right shape and the right magnitude. A model whose
worst case is qualitatively wrong is a different proposition, and Part 4 produces one.

### The learned basis

The trunk outputs $p$ functions of $x$, and every prediction is a linear combination of them. They
are not the sine modes, and there is no reason for them to be, but they span the same space of
solutions. Plotting a few of them shows what the network built.

In [ ]:
# --- A few trunk basis functions ---------------------------------------------
with torch.no_grad():
    T = torch.tanh(don.trunk(y_t)).numpy()          # (N_GRID, p)

# rank by how much each basis function is actually used across the test set
with torch.no_grad():
    Bco = don.branch(Qte_t).numpy()                 # (N_TEST, p)
use = np.abs(Bco).mean(0) * np.abs(T).max(0)
top = np.argsort(use)[::-1][:6]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for k in top:
    axes[0].plot(x_grid, T[:, k], lw=1.6, alpha=0.9)
axes[0].set_xlabel("$x$ (m)"); axes[0].set_ylabel("$t_k(x)$")
axes[0].set_title("six most used trunk basis functions", fontsize=10)

axes[1].semilogy(hist_don["train"], color=C_DATA, lw=1.4, label="train")
axes[1].semilogy(hist_don["test"],  color=C_BAD,  lw=1.4, label="test")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MSE (normalised units)")
axes[1].set_title(f"DeepONet training, {t_don:.0f} s", fontsize=10)
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: six of the $p$ trunk basis functions $t_k(x)$, chosen as
the ones the branch coefficients actually weight most heavily across the test set. Right: the
training and test mean squared error against epoch, in normalised units.

The basis functions are smooth, they vanish or nearly vanish at the supports, and they are not sine
modes. Nothing asked them to be. The network was free to choose any basis that spans the solutions
it was shown, and it chose one. In the right panel the two curves stay together, so the model is not
overfitting: with 2000 exact training pairs and no noise, there is nothing to overfit to.

### Branch coefficients times trunk functions, one term at a time

The prediction is the inner product from the top of this part,

$$G_\theta(q)(y) \;=\; \sum_{k=1}^{p} b_k\,t_k(y) \;+\; b_0, \qquad p = 48$$

The branch supplies the $p$ numbers $b_k$ for this particular load and nothing else. The trunk
supplies the $p$ functions $t_k(y)$ and never sees the load at all. The animation adds the terms one
at a time for a single test case, largest contribution first.

- Left: the trunk function being added, in orange, with the ones already added in grey. Its branch
  coefficient $b_k$ is in the title.
- Middle: the running sum, against the exact deflection and against the finished prediction.
- Right: how far the running sum still is from the finished prediction.

This is the whole architecture, made arithmetic.

In [ ]:
# --- Animation: the DeepONet sum assembled term by term ----------------------
x_q = np.linspace(0, L, 201)
_e_don_all = np.array([np.linalg.norm(P_don[i] - W_te[i]) / np.linalg.norm(W_te[i])
                       for i in range(N_TEST)])
i_demo = int(np.argsort(_e_don_all)[N_TEST // 2])   # a median-accuracy test case
a_demo_don = A_te[i_demo]

q_sens = torch.tensor(q_from_coeffs(a_demo_don, x_grid) / Q_SCALE, dtype=torch.float32)
y_q    = torch.tensor(x_q.reshape(-1, 1), dtype=torch.float32)
with torch.no_grad():
    b_coef = don.branch(q_sens).numpy().ravel()          # (p,)   branch output
    T_q    = torch.tanh(don.trunk(y_q)).numpy()          # (201, p)  trunk basis
    b0_val = don.b0.item()
    pred_don = don(q_sens, y_q).numpy().ravel() * W_SCALE
p_lat = len(b_coef)
TERMS = T_q * b_coef[None, :] * W_SCALE                  # (201, p) each term, in m
w_exact_q = w_from_coeffs(a_demo_don, x_q).ravel()

print(f"identity check, |sum_k b_k t_k + b_0 - prediction|_max = "
      f"{np.abs(TERMS.sum(1) + b0_val*W_SCALE - pred_don).max():.2e} m")
ordk = np.argsort(np.abs(TERMS).max(0))[::-1]            # largest contribution first
PS = np.hstack([np.full((len(x_q), 1), b0_val * W_SCALE),
                b0_val * W_SCALE + np.cumsum(TERMS[:, ordk], axis=1)])
err_ps = np.array([np.linalg.norm(PS[:, k] - pred_don) / np.linalg.norm(pred_don)
                   for k in range(p_lat + 1)])
print(f"peak exact deflection for this load : {np.abs(w_exact_q).max():.6f} m")
print(f"largest single term b_k t_k(x)      : {np.abs(TERMS).max():.6f} m")
for k in [1, 4, 12, 24, 48]:
    print(f"  {k:3d} terms: running sum is {err_ps[k]:.2e} away from the full prediction")
print(f"frames: {p_lat}")

T_LO, T_HI = T_q.min() * 1.15, T_q.max() * 1.15
lo = min(PS.min(), pred_don.min(), w_exact_q.min())
hi = max(PS.max(), pred_don.max(), w_exact_q.max())
pad = 0.10 * (hi - lo); lo -= pad; hi += pad
kax = np.arange(p_lat + 1)

fig, (d1, d2, d3) = plt.subplots(1, 3, figsize=(14.5, 4.0), gridspec_kw={"wspace": 0.30})

grey = [d1.plot([], [], color="0.78", lw=0.9)[0] for _ in range(p_lat)]
(dcur,) = d1.plot([], [], color=C_FIT, lw=2.4)
d1.axhline(0, color="k", lw=0.8)
d1.set_xlim(0, L); d1.set_ylim(T_LO, T_HI)
d1.set_xlabel("$x$  (m)"); d1.set_ylabel("$t_k(x)$  (dimensionless)")
td1 = d1.set_title("", fontsize=10)

d2.plot(x_q, w_exact_q, color=C_ALT, lw=3, alpha=0.6, label="exact")
d2.plot(x_q, pred_don, color="0.35", lw=1.3, ls=":", label="full DeepONet, all 48 terms")
(dsum,) = d2.plot([], [], color=C_BAD, lw=2, ls="--", label="running sum")
d2.set_xlim(0, L); d2.set_ylim(hi, lo)
d2.set_xlabel("$x$  (m)"); d2.set_ylabel("$w$  (m)")
d2.legend(fontsize=7.5, loc="lower center")
td2 = d2.set_title("", fontsize=10)

d3.semilogy(kax, np.maximum(err_ps, 1e-8), "o-", color=C_BAD, lw=1.4, ms=3.5)
(dmk,) = d3.plot([], [], "o", color="k", ms=9, mfc="none", mew=1.6)
d3.set_xlim(-0.5, p_lat + 0.5); d3.set_ylim(5e-9, 3)
d3.set_xlabel("terms included, $k$")
d3.set_ylabel("distance from the full prediction")
d3.set_title("the sum closing on the prediction", fontsize=10)

def update_don(f):
    k = f + 1
    for j in range(p_lat):
        if j < k:
            grey[j].set_data(x_q, T_q[:, ordk[j]])
        else:
            grey[j].set_data([], [])
    j = ordk[k - 1]
    dcur.set_data(x_q, T_q[:, j])
    dsum.set_data(x_q, PS[:, k])
    dmk.set_data([k], [max(err_ps[k], 6e-9)])
    td1.set_text("trunk function %d of %d,   branch coefficient $b_k$ = %+.3f"
                 % (k, p_lat, b_coef[j]))
    td2.set_text("%d terms,  distance from the full prediction = %.2e" % (k, err_ps[k]))
    return grey + [dcur, dsum, dmk]

anim_don = animation.FuncAnimation(fig, update_don, frames=p_lat, interval=150, blit=False)
plt.close(fig)
HTML(anim_don.to_jshtml())


**What the animation showed.** The running sum in the middle panel takes on the shape of the
deflection early and then closes on it steadily rather than in one jump, with the distances at 1, 4,
12, 24 and 48 terms printed above. The right panel is that same distance on a logarithmic axis. It
reaches the prediction only when the last term is added, which is what a sum of 48 terms with no
ordering built into it should do.

Two things follow from the structure rather than from this particular load. The trunk functions in
the left panel are the same 48 functions for every load the operator will ever see: they were fixed
when training stopped. Only the coefficients in the titles change from one load to the next. That is
why a new load case costs one forward pass of the branch and no solve.

The largest single term, printed above, is smaller than the peak deflection. That is worth
contrasting with the PINN in Notebook 5, where the same decomposition of the trained network gave
individual terms far larger than the answer, and the answer appeared only when they cancelled.

In [ ]:
# --- What are the trunk functions, measured rather than asserted -------------
# 1. Is any single trunk function close to a single sine mode?
def unitv(v): return v / np.linalg.norm(v)

best_mode, best_ov = np.zeros(p_lat, dtype=int), np.zeros(p_lat)
for k in range(p_lat):
    tk = T_q[:, k] - T_q[:, k].mean()
    if np.linalg.norm(tk) < 1e-12:
        continue
    ov = [abs(np.dot(unitv(tk), unitv(np.sin(n * np.pi * x_q / L)))) for n in range(1, 21)]
    j = int(np.argmax(ov)); best_mode[k], best_ov[k] = j + 1, ov[j]

k_best = int(np.argmax(best_ov))
print("1. correlation of each trunk function with the closest of sine modes 1 to 20")
print(f"   best   : {best_ov.max():.3f}  (trunk {k_best}, mode {best_mode[k_best]})")
print(f"   median : {np.median(best_ov):.3f}")
print(f"   number above 0.99 : {(best_ov > 0.99).sum()} of {p_lat}")
print("   So no trunk function is a sine mode. They are not meant to be.")
print()

# 2. Does the SPAN of the trunk contain the sine modes the training family is built from?
BASIS = np.hstack([np.ones((len(x_q), 1)), T_q])
Qo, _ = np.linalg.qr(BASIS)
print("2. fraction of each sine mode captured by the span of {1, t_1 ... t_p}")
print(f"{'mode n':>8s} {'captured':>10s}")
for n in range(1, N_MODES + 1):
    phi = unitv(np.sin(n * np.pi * x_q / L))
    print(f"{n:8d} {np.linalg.norm(Qo.T @ phi):10.5f}")
print("   1.00000 means the mode lies in the span to the precision shown.")
print()

# 3. How many independent directions are there really?
sv = np.linalg.svd(T_q, compute_uv=False)
rank = int((sv > 1e-3 * sv[0]).sum())
print("3. singular values of the 201 x 48 trunk matrix")
print("   " + "  ".join(f"{v:.2e}" for v in sv[:8]))
print(f"   directions above 1e-3 of the largest : {rank} of {p_lat}")

fig, (h1, h2) = plt.subplots(1, 2, figsize=(11.5, 3.9))
h1.semilogy(np.arange(1, p_lat + 1), np.maximum(sv, 1e-16), "o-", color=C_DATA, lw=1.4, ms=4)
h1.axhline(1e-3 * sv[0], color=C_BAD, ls="--", lw=1.2,
           label=f"$10^{{-3}}$ of the largest: {rank} above it")
h1.set_xlabel("index"); h1.set_ylabel("singular value")
h1.set_title("the 48 trunk functions, ranked", fontsize=10)
h1.legend(fontsize=8)

h2.bar(np.arange(1, p_lat + 1), best_ov, color=C_FIT, width=0.8)
h2.axhline(1.0, color="k", lw=0.8)
h2.set_ylim(0, 1.08)
h2.set_xlabel("trunk function $k$")
h2.set_ylabel("best correlation with one sine mode")
h2.set_title(f"closest single sine mode, best is {best_ov.max():.2f}", fontsize=10)
plt.tight_layout(); plt.show()


**What the two panels show, and what they do not.** Left: the singular values of the matrix
of trunk functions sampled on 201 points, with the number above one thousandth of the largest marked.
Right: for each trunk function, its correlation with the closest of the first twenty sine modes.

Three measured statements, and nothing beyond them.

First, no individual trunk function is a sine mode. The best correlation across all 48 is printed
above and in the right-hand title, and none reaches 0.99.

Second, the span of the trunk functions does contain the sine modes the training family is built
from. The captured fractions printed for modes 1 to 8 are all 1.00000 to the precision shown. The
network found the right subspace without being told what it was.

Third, the 48 functions are far from independent. Only the number printed in the left-hand legend
have singular values above a thousandth of the largest. Add the constant term to those directions
and the total matches the eight modes in the training family. Read that as consistent with the
second statement rather than as proof of anything. The practical point stands either way: the latent
dimension $p = 48$ is generous, most of it is redundant, and $p$ is usually chosen by trying a few
values rather than by an argument.

---

# Part 3: Fourier Neural Operator

### The idea

The FNO takes a different route. Instead of separating the input function from the query coordinate,
it works on the grid throughout and does its mixing in Fourier space.

Write $v_\ell(x)$ for the hidden state at layer $\ell$, a vector of $d_v$ channels at each point.
One layer does two things in parallel and adds them:

$$\boxed{\;v_{\ell+1}(x) \;=\; \sigma\Big(\underbrace{W_\ell\, v_\ell(x) + c_\ell}_{\text{pointwise}}
\;+\; \underbrace{F^{-1}\big[\,R_\ell \cdot F[v_\ell]\,\big](x)}
_{\text{spectral}}\Big)\;}$$

with the spectral path written mode by mode, for $k = 0, 1, \ldots$ :

$$\big(R_\ell \cdot \hat{v}_\ell\big)_{o,k} \;=\;
\begin{cases}
\displaystyle\sum_{i=1}^{d_v} \big(R_\ell\big)_{i,o,k}\,\big(\hat{v}_\ell\big)_{i,k},
& k < k_{\max}\\[2mm]
0, & k \ge k_{\max}
\end{cases}
\qquad \hat{v}_\ell = F[v_\ell]$$

Every symbol: $F$ and $F^{-1}$ are the discrete Fourier transform along $x$ and
its inverse; $\hat{v}_\ell$ are the Fourier coefficients of the hidden state; $k$ is the Fourier mode
index and $k_{\max} = 12$ is where the sum is truncated; $R_\ell$ is a learned **complex** weight
tensor of shape $d_v \times d_v \times k_{\max}$; $i$ and $o$ are the input and output channel
indices; $W_\ell$ is a real $d_v \times d_v$ matrix applied identically at every grid point, a
$1\times1$ convolution, with bias $c_\ell$; and $\sigma$ is the GELU activation. Here $d_v = 24$ and
there are three such layers.

The full network lifts the input to $d_v$ channels, applies these layers, and projects back down to
one channel.

Two consequences follow from the structure. Multiplication in Fourier space is convolution in space,
so the spectral path is a global convolution with a learned kernel, reaching the whole beam in one
layer rather than a receptive field at a time. And truncating at $k_{\max}$ is a built-in smoothing
that suits this problem, because the beam is itself a low-pass filter. The learned weights are
indexed by Fourier mode rather than by grid node, so nothing in the parameter set is tied to the
grid. Part 5 tests what that is worth.

In [ ]:
# --- Schematic: one FNO layer -----------------------------------------------
fig, ax = plt.subplots(figsize=(11.5, 4.4))
ax.set_xlim(0, 11.5); ax.set_ylim(0, 4.4); ax.axis("off"); ax.grid(False)

def fbox(x, y, w_, h_, label, col, fs=9.5):
    ax.add_patch(plt.Rectangle((x, y), w_, h_, fc="w", ec=col, lw=1.8, zorder=3))
    ax.text(x + w_/2, y + h_/2, label, ha="center", va="center", fontsize=fs, zorder=4)

def farrow(x0, y0, x1, y1, col="k"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=col, lw=1.5))

ax.text(0.55, 2.20, r"$v_\ell(x)$", ha="center", va="center", fontsize=13)
farrow(0.95, 2.55, 1.60, 3.30)          # up into the spectral path
farrow(0.95, 1.85, 1.60, 1.10)          # down into the pointwise path

# --- spectral path, top ------------------------------------------------------
fbox(1.60, 2.95, 1.25, 0.80, r"$F$", C_DATA, 13)
farrow(2.85, 3.35, 3.45, 3.35, C_DATA)
fbox(3.45, 2.95, 2.55, 0.80, "keep $k < k_{max}$\n" + r"multiply by $R_\ell$", C_DATA)
farrow(6.00, 3.35, 6.60, 3.35, C_DATA)
fbox(6.60, 2.95, 1.25, 0.80, r"$F^{-1}$", C_DATA, 13)
ax.text(4.72, 4.02, "spectral path: a global convolution, learned per Fourier mode",
        ha="center", fontsize=9, color=C_DATA)
farrow(7.85, 3.35, 8.75, 2.45, C_DATA)

# --- pointwise path, bottom --------------------------------------------------
fbox(3.45, 0.30, 2.55, 0.80, r"$W_\ell\,v_\ell(x) + c_\ell$", C_FIT)
farrow(1.60, 0.70, 3.45, 0.70, C_FIT)
farrow(6.00, 0.70, 8.75, 1.75, C_FIT)
ax.text(4.72, 0.00, "pointwise path: the same linear map at every grid point",
        ha="center", fontsize=9, color=C_FIT)

# --- sum and activation ------------------------------------------------------
ax.add_patch(plt.Circle((9.00, 2.10), 0.40, fc="w", ec="k", lw=1.8, zorder=3))
ax.text(9.00, 2.10, "+", ha="center", va="center", fontsize=17, zorder=4)
farrow(9.40, 2.10, 9.80, 2.10)
fbox(9.80, 1.70, 0.70, 0.80, r"$\sigma$", "k", 13)
ax.text(11.00, 2.10, r"$v_{\ell+1}(x)$", ha="center", va="center", fontsize=11)
ax.set_title("One FNO layer: two parallel paths meeting at a sum", fontsize=11)
plt.tight_layout(); plt.show()


**What the schematic shows.** The hidden state $v_\ell$ splits into two paths. The upper path
transforms to Fourier space, keeps the lowest $k_{\max}$ modes and multiplies them by the learned
complex weights $R_\ell$, throws the rest away, and transforms back. The lower path applies the same
small matrix $W_\ell$ at every grid point, touching no neighbours at all. They meet at the sum, and
the activation $\sigma$ closes the layer. The symbols are the ones in the boxed equation above.

The division of labour is the point. The spectral path is global and smooth, and it cannot represent
anything above mode $k_{\max}$. The pointwise path is local and keeps whatever sharp, node-level
detail the spectral truncation discarded. Neither path alone would do.

In [ ]:
# --- Spectral convolution and the FNO ----------------------------------------
class SpectralConv1d(nn.Module):
    '''Multiply the lowest `modes` Fourier coefficients by learned complex weights.'''
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        scale = 1.0 / (in_ch * out_ch)
        # last axis of size 2 holds the real and imaginary parts
        self.weight = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, 2))

    def forward(self, v):                      # v: (B, in_ch, N)
        B, _, N = v.shape
        v_ft = torch.fft.rfft(v, norm="forward")           # (B, in_ch, N//2+1)
        R = torch.view_as_complex(self.weight)             # (in_ch, out_ch, modes)
        k = min(self.modes, v_ft.shape[-1])
        out_ft = torch.zeros(B, self.out_ch, v_ft.shape[-1],
                             dtype=torch.cfloat, device=v.device)
        out_ft[:, :, :k] = torch.einsum("bik,iok->bok", v_ft[:, :, :k], R[:, :, :k])
        return torch.fft.irfft(out_ft, n=N, norm="forward")

class FNO1d(nn.Module):
    def __init__(self, modes=12, width=24, n_layers=3):
        super().__init__()
        self.lift = nn.Conv1d(2, width, 1)                 # channels: [q(x), x]
        self.spec = nn.ModuleList([SpectralConv1d(width, width, modes)
                                   for _ in range(n_layers)])
        self.pw   = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Conv1d(width, 64, 1), nn.GELU(), nn.Conv1d(64, 1, 1))

    def forward(self, q):                                  # q: (B, N) normalised
        B, N = q.shape
        xc = torch.linspace(0, 1, N, device=q.device).expand(B, N)
        v = self.lift(torch.stack([q, xc], dim=1))
        for s, w in zip(self.spec, self.pw):
            v = torch.nn.functional.gelu(s(v) + w(v))
        return self.proj(v).squeeze(1)                     # (B, N)

torch.manual_seed(0)
fno = FNO1d()
print(f"FNO: width 24, 12 modes, 3 spectral layers")
print(f"parameters: {sum(p.numel() for p in fno.parameters())}")
print(f"DeepONet had {sum(p.numel() for p in don.parameters())}")


### What the spectral layer keeps, and what it throws away

Before training it, look at the truncation on its own. The spectral path takes the discrete Fourier
transform of the hidden state along $x$, keeps the coefficients with $k < k_{\max}$, and discards
the rest. Whatever the learned weights $R_\ell$ do, they can only act on what survives that cut.

The animation reconstructs two loads from an increasing number of Fourier coefficients on the same
64-point grid the operator uses, with the exact function underneath and the relative $L_2$ error in
each title. Both are transformed with `numpy.fft.rfft`, the same transform `SpectralConv1d` calls,
so the mode index on the horizontal axis of the right panel is the network's own $k$.

Note that this is a different basis from the sine series of Part 1. The sine series was the
eigenbasis of the beam operator and made the solve exact. The discrete Fourier transform here is
whatever the FNO happens to use, and a load built from eight sine modes is not built from eight
Fourier coefficients.

- Left: a load from the training distribution, eight sine modes with $1/n$ decay.
- Middle: a step load, constant over the middle third. A step is not band limited, and its Fourier
  coefficients fall off only as $1/k$.
- Right: the error against the number of coefficients kept, with the shaded region marking the
  coefficients the spectral layer never sees.

In [ ]:
# --- Animation: spectral truncation, and where the FNO cuts ------------------
K_MAX = fno.spec[0].modes
N_BINS = N_GRID // 2 + 1

rng_tr = np.random.default_rng(3)
a_demo = sample_coeffs(1, rng_tr).ravel()
q_smooth = q_from_coeffs(a_demo, x_grid).ravel()
q_step64 = np.where((x_grid > L/3) & (x_grid < 2*L/3), 1.0, 0.0)

def truncate_fft(f, K):
    '''Keep the lowest K rfft coefficients of f and transform back. The same cut
       SpectralConv1d makes, without the learned weights.'''
    F = np.fft.rfft(f)
    G = np.zeros_like(F); G[:K] = F[:K]
    return np.fft.irfft(G, n=len(f))

Ks = np.arange(1, N_BINS + 1)
R_sm = np.array([truncate_fft(q_smooth, K) for K in Ks])
R_st = np.array([truncate_fft(q_step64, K) for K in Ks])
e_sm = np.array([np.linalg.norm(r - q_smooth) / np.linalg.norm(q_smooth) for r in R_sm])
e_st = np.array([np.linalg.norm(r - q_step64) / np.linalg.norm(q_step64) for r in R_st])

print(f"the 64-point grid carries {N_BINS} rfft coefficients; the FNO keeps {K_MAX}")
print(f"reconstruction error at the FNO cut, K = {K_MAX}:")
print(f"   8-mode training load : {e_sm[K_MAX-1]:.3e}")
print(f"   step load            : {e_st[K_MAX-1]:.3e}")
print()
print("step load, how slowly it improves, and the overshoot at the plateau:")
print(f"{'K':>6s} {'rel L2':>10s} {'max value':>11s}")
for K_ in [4, 8, K_MAX, 20, 28]:
    print(f"{K_:6d} {e_st[K_-1]:10.3e} {R_st[K_-1].max():11.3f}")
print("The exact plateau is 1.000. The reconstruction still overshoots it beside each")
print("jump at every K shown, even when most of the coefficients are kept.")
print(f"frames: {len(Ks)}")

FLOOR = 1e-8
fig, (f1, f2, f3) = plt.subplots(1, 3, figsize=(14.5, 4.0), gridspec_kw={"wspace": 0.30})
for ax, f, R, ttl in [(f1, q_smooth, R_sm, "training load: 8 sine modes"),
                      (f2, q_step64, R_st, "step load: not band limited")]:
    ax.plot(x_grid, f, color=C_ALT, lw=3, alpha=0.6, label="exact, on the 64-point grid")
    ax.set_xlim(0, L)
    lo, hi = min(R.min(), f.min()), max(R.max(), f.max())
    pad = 0.12 * (hi - lo)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel("$x$  (m)"); ax.set_ylabel("$q$  (N/m)")
(lf1,) = f1.plot([], [], color=C_BAD, lw=1.8, ls="--", label="truncated")
(lf2,) = f2.plot([], [], color=C_BAD, lw=1.8, ls="--", label="truncated")
f1.legend(fontsize=7.5, loc="lower right"); f2.legend(fontsize=7.5, loc="lower right")
tf1 = f1.set_title("", fontsize=10); tf2 = f2.set_title("", fontsize=10)

f3.semilogy(Ks, np.maximum(e_sm, FLOOR), "o-", color=C_DATA, lw=1.5, ms=3.5,
            label="8-mode training load")
f3.semilogy(Ks, np.maximum(e_st, FLOOR), "s-", color=C_BAD, lw=1.5, ms=3.5,
            label="step load")
f3.axvspan(K_MAX, N_BINS + 0.5, color="0.6", alpha=0.16)
f3.axvline(K_MAX, color="k", ls="--", lw=1.2)
f3.text(N_BINS * 0.70, 5.0,
        "shaded: coefficients the spectral\nlayer never sees, $k \\geq k_{max} = %d$" % K_MAX,
        fontsize=8, ha="center", va="top",
        bbox=dict(fc="w", ec="0.7", alpha=0.92, boxstyle="round,pad=0.25"))
(mkf,) = f3.plot([], [], "o", color="k", ms=9, mfc="none", mew=1.6)
f3.set_xlim(0.5, N_BINS + 0.5); f3.set_ylim(5e-9, 6)
f3.set_xlabel("Fourier coefficients kept, $K$"); f3.set_ylabel("relative $L_2$ error")
f3.legend(fontsize=8, loc="lower left")
f3.set_title("what truncation costs", fontsize=10)

def update_fft(i):
    K = Ks[i]
    lf1.set_data(x_grid, R_sm[i]); lf2.set_data(x_grid, R_st[i])
    tf1.set_text("$K$ = %d of %d kept,  error = %.2e" % (K, N_BINS, e_sm[i]))
    tf2.set_text("$K$ = %d of %d kept,  error = %.2e" % (K, N_BINS, e_st[i]))
    mkf.set_data([K, K], [max(e_sm[i], 6e-9), max(e_st[i], 6e-9)])
    return lf1, lf2, mkf

anim_fft = animation.FuncAnimation(fig, update_fft, frames=len(Ks), interval=170, blit=False)
plt.close(fig)
HTML(anim_fft.to_jshtml())


**What the animation showed.** The smooth load is recovered to the accuracy printed above
well before the cut. Its Fourier coefficients decay fast, so the coefficients the spectral layer
discards carry very little: the error printed at $K = k_{\max} = 12$ is well under one percent.

The step load never recovers. The reconstruction oscillates near the two jumps and overshoots the
plateau by several percent at every $K$ in the printed table. That is
the Gibbs phenomenon, and every reader who has run a spectral method has seen it. The printed table
shows both halves of it: the error falls slowly and steadily as coefficients are added, while the
overshoot beside each jump stays where it is.

Two consequences for the rest of the notebook. First, the truncation is a reasonable design choice
here because the beam attenuates mode $n$ by $n^{-4}$, so detail the spectral path discards was
never going to reach the deflection anyway. Second, the FNO has no representation at all of what
lies beyond mode $k_{\max}$ in the input. Part 4 feeds it a step load and a single high mode, and
this animation is why those cases go wrong.

In [ ]:
# --- Train the FNO on exactly the same data ----------------------------------
EPOCHS_FNO = 120

opt = torch.optim.Adam(fno.parameters(), lr=3e-3)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=40, gamma=0.3)

hist_fno = {"train": [], "test": []}
t0 = time.time()
for ep in range(EPOCHS_FNO):
    perm = torch.randperm(N_TRAIN)
    running = 0.0
    for i in range(0, N_TRAIN, BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad()
        loss = lossf(fno(Qtr_t[idx]), Wtr_t[idx])
        loss.backward(); opt.step()
        running += loss.item() * len(idx)
    sched.step()
    with torch.no_grad():
        te = lossf(fno(Qte_t), Wte_t).item()
    hist_fno["train"].append(running / N_TRAIN); hist_fno["test"].append(te)
    if (ep + 1) % 30 == 0:
        print(f"  epoch {ep+1:4d}   train MSE {running/N_TRAIN:.3e}   "
              f"test MSE {te:.3e}", flush=True)
t_fno = time.time() - t0

with torch.no_grad():
    P_fno = fno(Qte_t).numpy() * W_SCALE
err_fno = rel_l2_rows(P_fno, W_te)
print(f"\nFNO trained in {t_fno:.1f} s")
print(f"mean relative L2 error on the same {N_TEST} unseen loads: {err_fno:.4e}")


In [ ]:
# --- Head to head on the same test set ---------------------------------------
e_don = np.array([np.linalg.norm(P_don[i]-W_te[i])/np.linalg.norm(W_te[i])
                  for i in range(N_TEST)])
e_fno = np.array([np.linalg.norm(P_fno[i]-W_te[i])/np.linalg.norm(W_te[i])
                  for i in range(N_TEST)])

print(f"{'':12s} {'mean':>10s} {'median':>10s} {'worst':>10s} {'train time':>12s} {'params':>9s}")
for nm, e, t, m in [("DeepONet", e_don, t_don, don), ("FNO", e_fno, t_fno, fno)]:
    print(f"{nm:12s} {e.mean():10.3e} {np.median(e):10.3e} {e.max():10.3e} "
          f"{t:11.1f} s {sum(p.numel() for p in m.parameters()):9d}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
bins = np.logspace(np.log10(min(e_don.min(), e_fno.min())),
                   np.log10(max(e_don.max(), e_fno.max())), 35)
axes[0].hist(e_don, bins=bins, alpha=0.6, color=C_DATA, label="DeepONet")
axes[0].hist(e_fno, bins=bins, alpha=0.6, color=C_FIT, label="FNO")
axes[0].set_xscale("log"); axes[0].set_xlabel("relative $L_2$ error")
axes[0].set_ylabel("test cases"); axes[0].set_title("error distribution", fontsize=10)
axes[0].legend(fontsize=8)

axes[1].semilogy(hist_don["test"], color=C_DATA, lw=1.4, label="DeepONet")
axes[1].semilogy(hist_fno["test"], color=C_FIT,  lw=1.4, label="FNO")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("test MSE")
axes[1].set_title("different epoch counts, same data", fontsize=10); axes[1].legend(fontsize=8)

k = int(np.argmax(e_don))
axes[2].plot(x_grid, W_te[k], color=C_ALT, lw=3, alpha=0.6, label="exact")
axes[2].plot(x_grid, P_don[k], color=C_DATA, lw=1.5, ls="--", label="DeepONet")
axes[2].plot(x_grid, P_fno[k], color=C_FIT,  lw=1.5, ls=":",  label="FNO")
axes[2].invert_yaxis(); axes[2].set_xlabel("$x$ (m)"); axes[2].set_ylabel("$w$ (m)")
axes[2].set_title("the DeepONet's worst test case", fontsize=10); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()


**What the three panels show.** Left: the distribution of per-case relative $L_2$ error over
the 400 test loads, on a logarithmic axis, one histogram per model. Middle: the test mean squared
error against epoch for both, noting that the two were trained for different epoch counts. Right:
the single test case the DeepONet handles worst, with both models drawn against the exact solution.

The right panel is the useful one. Even on the case the DeepONet handles worst, the predicted curve
has the right shape and magnitude. Compare that with the failures in Part 4, where the shape itself
goes wrong. An error of a few percent on the hardest case in distribution is a different kind of
problem from being outside the distribution altogether.

Read the table rather than the architecture names. Both models reach a small error on this
problem, and the gap between them is not the interesting part: the interesting part is that either
one covers the whole family after a single training run.

### How the two models are queried from here on

The next cell fixes one protocol and uses it everywhere below, because the comparison in Part 4 is
only meaningful if both models are asked the same question. Both are evaluated on the 64-point grid
they were trained on, and the result is interpolated to the plotting grid when a picture needs it.
Part 5 changes the grid on purpose and measures what that costs.

In [ ]:
# --- Prediction helpers, and the evaluation protocol -------------------------
# Both operators are asked for a prediction ON THE 64-POINT TRAINING GRID and then
# interpolated to the plotting grid. That keeps every accuracy number in this notebook
# comparable with every other one, and it keeps accuracy and timing on the same grid.
# Part 5 is where the FNO is deliberately evaluated on other grids, and it measures
# what that costs, so doing it silently here would contaminate the comparison.

x_plot = np.linspace(0, L, 201)

def predict_don(a_vec, xq=None):
    '''DeepONet: branch reads the 64 sensors, trunk is queried at xq.'''
    xq = x_grid if xq is None else xq
    qs = torch.tensor(q_from_coeffs(a_vec, x_grid) / Q_SCALE, dtype=torch.float32)
    yq = torch.tensor(xq.reshape(-1, 1), dtype=torch.float32)
    with torch.no_grad():
        return don(qs, yq).numpy().ravel() * W_SCALE

def predict_fno(a_vec, xq=None):
    '''FNO: predicts on the 64-point grid, then interpolates to xq.'''
    qs = torch.tensor(q_from_coeffs(a_vec, x_grid) / Q_SCALE, dtype=torch.float32)
    with torch.no_grad():
        w64 = fno(qs).numpy().ravel() * W_SCALE
    return w64 if xq is None else np.interp(xq, x_grid, w64)


### Set the load yourself

The sliders below set $a_1$ to $a_5$ directly. Every move is one forward pass, so the response is
immediate, and the exact answer is available for comparison because the series solution is closed
form. Try to find a combination the operator gets wrong.

In [ ]:
# --- Interactive: build your own load ----------------------------------------
def explore(a1=1.0, a2=0.0, a3=0.0, a4=0.0, a5=0.0, model="DeepONet"):
    a = np.zeros(N_MODES); a[:5] = [a1, a2, a3, a4, a5]
    q_p = q_from_coeffs(a, x_plot).ravel()
    w_e = w_from_coeffs(a, x_plot).ravel()
    if model == "DeepONet":
        w_p = predict_don(a, x_plot)
    else:
        w_p = predict_fno(a, x_plot)
    nrm = np.linalg.norm(w_e)
    err = np.linalg.norm(w_p - w_e) / nrm if nrm > 1e-14 else np.nan

    fig, (a1x, a2x) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    a1x.plot(x_plot, q_p, color=C_DATA, lw=2)
    a1x.axhline(0, color="k", lw=0.8)
    a1x.set_xlabel("$x$ (m)"); a1x.set_ylabel("$q$ (N/m)")
    a1x.set_title("the load you built", fontsize=10)

    a2x.plot(x_plot, w_e, color=C_ALT, lw=3, alpha=0.6, label="exact")
    a2x.plot(x_plot, w_p, color=C_BAD, lw=1.8, ls="--", label=model)
    a2x.invert_yaxis(); a2x.set_ylim(0.06, -0.06)
    a2x.set_xlabel("$x$ (m)"); a2x.set_ylabel("$w$ (m)")
    a2x.set_title(f"relative $L_2$ error = {err:.2e}", fontsize=10)
    a2x.legend(fontsize=8, loc="lower center")
    plt.tight_layout(); plt.show()

sl = lambda v: FloatSlider(v, min=-2.0, max=2.0, step=0.1, continuous_update=False)
interact(explore, a1=sl(1.0), a2=sl(0.0), a3=sl(0.0), a4=sl(0.0), a5=sl(0.0),
         model=Dropdown(options=["DeepONet", "FNO"], value="DeepONet"));


### A load that moves

Sweeping one parameter continuously makes the point better than a static figure. The load below is a
band-limited travelling bump: its sine coefficients are $a_n = (2/n)\sin(n\pi s/L)$, which moves the
peak to position $s$ while keeping the $1/n$ decay of the training distribution.

Each frame is one forward pass. Sixty frames of a parametric study, in the time it takes to draw
them.

In [ ]:
# --- Animation: a load travelling along the beam -----------------------------
N_FRAMES = 60
s_vals = np.linspace(0.08, 0.92, N_FRAMES) * L
n_arr = np.arange(1, N_MODES + 1)

A_sweep = np.array([(2.0 / n_arr) * np.sin(n_arr * np.pi * s / L) for s in s_vals])
Q_sweep = q_from_coeffs(A_sweep, x_plot)
W_sweep = w_from_coeffs(A_sweep, x_plot)
P_sweep = np.array([predict_don(A_sweep[i], x_plot) for i in range(N_FRAMES)])
err_sweep = np.array([np.linalg.norm(P_sweep[i]-W_sweep[i])/np.linalg.norm(W_sweep[i])
                      for i in range(N_FRAMES)])
print(f"over {N_FRAMES} load positions: mean relative L2 = {err_sweep.mean():.3e}, "
      f"worst = {err_sweep.max():.3e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))
(lq,) = ax1.plot([], [], color=C_DATA, lw=2)
ax1.set_xlim(0, L); ax1.set_ylim(Q_sweep.min()*1.1, Q_sweep.max()*1.1)
ax1.axhline(0, color="k", lw=0.8)
ax1.set_xlabel("$x$ (m)"); ax1.set_ylabel("$q$ (N/m)"); ax1.set_title("load", fontsize=10)

(le,) = ax2.plot([], [], color=C_ALT, lw=3, alpha=0.6, label="exact")
(lp,) = ax2.plot([], [], color=C_BAD, lw=1.8, ls="--", label="DeepONet")
ax2.set_xlim(0, L); ax2.set_ylim(W_sweep.max()*1.25, W_sweep.min()*1.25)
ax2.set_xlabel("$x$ (m)"); ax2.set_ylabel("$w$ (m)"); ax2.legend(fontsize=8, loc="lower center")
ttl = ax2.set_title("", fontsize=10)

def update(f):
    lq.set_data(x_plot, Q_sweep[f])
    le.set_data(x_plot, W_sweep[f]); lp.set_data(x_plot, P_sweep[f])
    ttl.set_text(f"peak at $s$ = {s_vals[f]:.2f} m    "
                 f"relative $L_2$ = {err_sweep[f]:.2e}")
    return lq, le, lp, ttl

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=90, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


**What the animation shows.** Left: the load, a bump whose peak travels from one end of the
beam to the other. Right: the deflection it produces, green for exact and red dashed for the
DeepONet, with the position of the peak and the relative $L_2$ error of that frame in the title.

Sixty frames is sixty load cases. The mean and worst error over the sweep are printed above the
animation, and the prediction tracks the exact curve throughout, including at the ends where the
load is pressed up against a support. Every frame of this cost one forward pass. In Notebook 5 every
frame would have cost a full retraining run.

---

# Part 4: The comparison, and the honest limitation

### Retrain against forward pass

Take one load the operator has never seen. Solve it two ways.

The PINN is the Notebook 5 recipe, unchanged: a 32-wide, 3-layer `tanh` network taking $x$ to $w$,
100 collocation points, Adam followed by L-BFGS, with the residual $EI\,w'''' - q$ and the four
boundary terms as the entire loss. It starts from a random initialisation because that is what
happens when the load changes. The boundary weight $\lambda$ was picked from a short sweep so that
the PINN reaches an accuracy comparable to the operator, because a timing comparison against a
badly converged baseline would prove nothing.

The operator does one forward pass.

State the asymmetry plainly before looking at the numbers. The operator required 2000 solved
problems before it could do anything, and the PINN required none. The comparison below is between a
model that has already paid that cost and one that pays per problem.

In [ ]:
# --- The PINN from Notebook 5, unchanged -------------------------------------
def d_dx(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

class BeamNet(nn.Module):
    def __init__(self, hidden=32, n_layers=3, seed=1):
        super().__init__()
        torch.manual_seed(seed)
        layers = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)

def derivatives(model, x, order=4):
    out = [model(x)]
    for _ in range(order):
        out.append(d_dx(out[-1], x))
    return out

_b = BeamNet()
print(f"PINN: 1 -> 32 -> 32 -> 32 -> 1, tanh, {sum(p.numel() for p in _b.parameters())} parameters")
print(f"DeepONet: {sum(p.numel() for p in don.parameters())} parameters")
print("The PINN is the smaller network. It is not losing on capacity.")


In [ ]:
# --- One unseen load, solved by PINN from scratch ----------------------------
LAMBDA_BC, N_COL, ADAM_ITERS = 1.0e3, 100, 2000

a_new = sample_coeffs(1, np.random.default_rng(99))[0]
print("coefficients of the test load:", np.array2string(a_new, precision=3))

a_t = torch.tensor(a_new, dtype=torch.float32)
n_t = torch.arange(1, N_MODES + 1, dtype=torch.float32)
def q_new_torch(x):
    return (torch.sin(n_t * np.pi * x / L) * a_t).sum(dim=1, keepdim=True)

# Every error below is measured on the 64-point grid, against the exact solution sampled
# there, for all three models. x_plot is used for drawing only.
w_new_exact_g64  = w_from_coeffs(a_new, x_grid).ravel()
w_new_exact_plot = w_from_coeffs(a_new, x_plot).ravel()
xg_t = torch.tensor(x_grid.reshape(-1, 1), dtype=torch.float32)
xp_t = torch.tensor(x_plot.reshape(-1, 1), dtype=torch.float32)

pinn = BeamNet(seed=1)
x_col = torch.linspace(0, L, N_COL).reshape(-1, 1).requires_grad_(True)
x_bc  = torch.tensor([[0.0], [L]]).requires_grad_(True)

def total_loss(m):
    d = derivatives(m, x_col, order=4)
    lp = torch.mean((EI * d[4] - q_new_torch(x_col)) ** 2)
    db = derivatives(m, x_bc, order=2)
    lb = torch.mean(db[0] ** 2) + torch.mean(db[2] ** 2)
    return lp + LAMBDA_BC * lb

t0 = time.time()
opt_p = torch.optim.Adam(pinn.parameters(), lr=2e-3)
for it in range(ADAM_ITERS):
    opt_p.zero_grad(); l = total_loss(pinn); l.backward(); opt_p.step()
opt_l = torch.optim.LBFGS(pinn.parameters(), lr=1.0, max_iter=250, history_size=50,
                          line_search_fn="strong_wolfe")
def closure():
    opt_l.zero_grad(); l = total_loss(pinn); l.backward(); return l
opt_l.step(closure)
t_pinn_retrain = time.time() - t0

with torch.no_grad():
    w_pinn     = pinn(xp_t).numpy().ravel()          # for the picture
    w_pinn_g64 = pinn(xg_t).numpy().ravel()          # for the error, same grid as the operators
err_pinn = (np.linalg.norm(w_pinn_g64 - w_new_exact_g64)
            / np.linalg.norm(w_new_exact_g64))
print(f"\nPINN retrained from scratch in {t_pinn_retrain:.1f} s   "
      f"relative L2 on the 64-point grid = {err_pinn:.3e}")


In [ ]:
# --- The same load, by one operator forward pass -----------------------------
q_one = torch.tensor(q_from_coeffs(a_new, x_grid) / Q_SCALE, dtype=torch.float32)

# warm up once so the timing is not measuring lazy initialisation
with torch.no_grad():
    don(q_one, y_t); fno(q_one)

REPS = 200
with torch.no_grad():
    t0 = time.time()
    for _ in range(REPS): don(q_one, y_t)
    t_don_fwd = (time.time() - t0) / REPS
    t0 = time.time()
    for _ in range(REPS): fno(q_one)
    t_fno_fwd = (time.time() - t0) / REPS

# Errors on the same 64-point grid that was just timed, for all three models.
w_don_g64 = predict_don(a_new)
w_fno_g64 = predict_fno(a_new)
nrm64 = np.linalg.norm(w_new_exact_g64)
err_don_new = np.linalg.norm(w_don_g64 - w_new_exact_g64)/nrm64
err_fno_new = np.linalg.norm(w_fno_g64 - w_new_exact_g64)/nrm64
w_don_new = np.interp(x_plot, x_grid, w_don_g64)      # for the picture only
w_fno_new = np.interp(x_plot, x_grid, w_fno_g64)

print("=" * 74)
print("  one unseen load, every error on the same 64-point grid")
print(f"  PINN, retrained from scratch : {t_pinn_retrain:10.2f} s     "
      f"relative L2 = {err_pinn:.2e}")
print(f"  DeepONet, one forward pass   : {t_don_fwd*1e3:10.3f} ms    "
      f"relative L2 = {err_don_new:.2e}")
print(f"  FNO, one forward pass        : {t_fno_fwd*1e3:10.3f} ms    "
      f"relative L2 = {err_fno_new:.2e}")
print("-" * 74)
print(f"  speed-up, PINN retrain / DeepONet pass : {t_pinn_retrain/t_don_fwd:,.0f}x")
print(f"  speed-up, PINN retrain / FNO pass      : {t_pinn_retrain/t_fno_fwd:,.0f}x")
print("-" * 74)
print(f"  up-front cost of each operator: {N_TRAIN} solved problems, plus training")
print(f"  up-front cost of the PINN     : none")
print(f"{'':32s} {'training':>10s} {'break-even vs the PINN':>24s}")
for nm, t_tr in [("DeepONet", t_don), ("FNO", t_fno)]:
    print(f"  {nm:30s} {t_tr:8.1f} s {t_tr/t_pinn_retrain:19.1f} load cases")
print("  Break-even is that model's own training time divided by one PINN solve, and it")
print("  ignores the cost of generating the 2000 training solutions, which on a real")
print("  problem is the dominant term.")
print("=" * 74)


In [ ]:
# --- The three solutions, drawn together -------------------------------------
fig, (a1x, a2x) = plt.subplots(1, 2, figsize=(11.5, 4))
a1x.plot(x_plot, q_from_coeffs(a_new, x_plot).ravel(), color=C_DATA, lw=2)
a1x.axhline(0, color="k", lw=0.8)
a1x.set_xlabel("$x$ (m)"); a1x.set_ylabel("$q$ (N/m)")
a1x.set_title("the unseen load", fontsize=10)

a2x.plot(x_plot, w_new_exact_plot, color=C_ALT, lw=3.5, alpha=0.5, label="exact")
a2x.plot(x_plot, w_pinn,    color=C_BAD,  lw=1.5, ls="-",  label=f"PINN, {t_pinn_retrain:.0f} s")
a2x.plot(x_plot, w_don_new, color=C_DATA, lw=1.5, ls="--", label=f"DeepONet, {t_don_fwd*1e3:.2f} ms")
a2x.plot(x_plot, w_fno_new, color=C_FIT,  lw=1.5, ls=":",  label=f"FNO, {t_fno_fwd*1e3:.2f} ms")
a2x.invert_yaxis(); a2x.set_xlabel("$x$ (m)"); a2x.set_ylabel("$w$ (m)")
a2x.set_title("same answer, very different cost", fontsize=10)
a2x.legend(fontsize=8, loc="lower center")
plt.tight_layout(); plt.show()


**What the two panels show.** Left: the unseen load. Right: four curves on top of each other,
the exact solution and three approximations, with the wall clock cost of each in the legend. The two
operator curves are their 64-point predictions interpolated for drawing, which is why the drawn
curves are smoother than the grid they came from.

All four curves lie on one another. The accuracies printed above are within a factor of a few of each
other and the costs differ by the factor in the table, which is the point. Two caveats sit in that
same table. The operators needed 2000 solved problems before any of this could happen, and the
break-even line counts only the training time, not the cost of producing those solutions.

One number is worth a second look. The FNO's error on this single load is well above its mean over
the 400 test loads, printed earlier. That is not a protocol artefact, it is one draw from a
distribution of per-case errors, and the histogram in Part 3 is how wide that distribution is. A
single load case is not a measurement of a model.

### Now break it

The operator was trained on loads containing modes 1 to 8 with $1/n$ decay. Two loads outside that
set:

- $q(x) = \sin(20\pi x/L)$, a single mode well above anything in the training data.
- A step load, constant over the middle third of the span and zero elsewhere. A step is not
  band-limited at all: its sine series has coefficients falling off only as $1/n$.

Both have exact reference solutions. For the single mode the series formula applies directly. For
the step the coefficients are computed by numerical integration with 200 modes, and that reference
is checked against a finite difference solve of the same boundary value problem before it is used.

In [ ]:
# --- Two out-of-distribution loads, with verified references -----------------
N_REF = 200                      # modes used for the reference solutions

def coeffs_of(q_fun, n_modes=N_REF, n_quad=8001):
    '''a_n = (2/L) int_0^L q(x) sin(n pi x / L) dx, by the trapezoid rule.'''
    xq = np.linspace(0, L, n_quad)
    nn_ = np.arange(1, n_modes + 1)[:, None]
    return (2.0 / L) * np.trapezoid(q_fun(xq)[None, :] * np.sin(nn_*np.pi*xq/L), xq, axis=1)

q_step_fun = lambda x: np.where((x > L/3) & (x < 2*L/3), 1.0, 0.0)
a_step = coeffs_of(q_step_fun)

a_hi = np.zeros(20); a_hi[19] = 1.0            # pure mode 20

# check the step reference against a finite difference BVP solve
q_b = q_step_fun(x_bvp)
u_b = np.zeros(n_bvp); u_b[1:-1] = np.linalg.solve(A, q_b[1:-1])
w_b = np.zeros(n_bvp); w_b[1:-1] = np.linalg.solve(A, u_b[1:-1] / EI)
w_step_series = w_from_coeffs(a_step, x_bvp).ravel()
print(f"step load: {N_REF}-mode series vs finite difference solve, relative L2 = "
      f"{np.linalg.norm(w_b - w_step_series)/np.linalg.norm(w_step_series):.3e}")
print("The deflection is smooth even though the load is not, because of the four integrations.")


In [ ]:
# --- How badly does the operator do outside its training distribution? -------
def ood_case(a_full, name):
    # same protocol as Part 4: both models predict on the 64-point grid, errors are
    # measured there against the exact solution sampled there, and the curves are
    # interpolated only for drawing.
    w_e64 = w_from_coeffs(a_full, x_grid).ravel()
    w_d64 = predict_don(a_full)
    w_f64 = predict_fno(a_full)
    nrm   = np.linalg.norm(w_e64)
    ed = np.linalg.norm(w_d64 - w_e64)/nrm
    ef = np.linalg.norm(w_f64 - w_e64)/nrm
    return dict(name=name,
                q=q_from_coeffs(a_full, x_plot).ravel(),
                we=w_from_coeffs(a_full, x_plot).ravel(),
                wd=np.interp(x_plot, x_grid, w_d64),
                wf=np.interp(x_plot, x_grid, w_f64),
                we64=w_e64, wd64=w_d64, wf64=w_f64, ed=ed, ef=ef)

# a typical in-distribution case, chosen as the median of the 400 test loads
i_med = int(np.argsort(e_don)[len(e_don)//2])
cases = [ood_case(A_te[i_med], "in distribution, modes 1-8"),
         ood_case(a_hi,        "mode 20 only"),
         ood_case(a_step,      "step load over the middle third")]

print("mean over the 400 in-distribution test loads:")
print(f"    DeepONet {err_don:.3e}     FNO {err_fno:.3e}")
print()
print(f"{'single load':38s} {'DeepONet':>11s} {'FNO':>11s} {'predicted/exact peak |w|':>26s}")
for c in cases:
    amp = np.abs(c["we64"]).max()
    ratio = f"{np.abs(c['wd64']).max()/amp:.1f} (DON), {np.abs(c['wf64']).max()/amp:.1f} (FNO)"
    print(f"{c['name']:38s} {c['ed']:11.3e} {c['ef']:11.3e} {ratio:>26s}")

fig, axes = plt.subplots(2, 3, figsize=(13.5, 6))
for j, c in enumerate(cases):
    axes[0, j].plot(x_plot, c["q"], color=C_DATA, lw=1.6)
    axes[0, j].set_title(c["name"], fontsize=9.5)
    axes[0, j].set_ylabel("$q$ (N/m)" if j == 0 else "")
    axes[1, j].plot(x_plot, c["we"], color=C_ALT, lw=3, alpha=0.6, label="exact")
    axes[1, j].plot(x_plot, c["wd"], color=C_DATA, lw=1.5, ls="--", label="DeepONet")
    axes[1, j].plot(x_plot, c["wf"], color=C_FIT,  lw=1.5, ls=":",  label="FNO")
    axes[1, j].invert_yaxis(); axes[1, j].set_xlabel("$x$ (m)")
    axes[1, j].set_ylabel("$w$ (m)" if j == 0 else "")
    axes[1, j].set_title(f"DeepONet {c['ed']:.1e}   FNO {c['ef']:.1e}", fontsize=9.5)
axes[1, 0].legend(fontsize=8, loc="lower center")
plt.tight_layout(); plt.show()


**What the six panels show.** One column per load: in distribution on the left, a single mode
far above the training range in the middle, a discontinuous step load on the right. Top row is the
load, bottom row is the deflection, with the exact solution in green and the two operators dashed
and dotted. The per-case errors are in each title, and the table above also prints the ratio of the
predicted peak deflection to the exact one.

Read those three columns carefully, because they are the whole caveat.

The operator is accurate on loads drawn from the distribution it was trained on. On a mode it has
never seen, the relative error rises by orders of magnitude and the predicted shape is wrong. The
step load sits in between, because most of its energy is in the low modes that are in distribution,
but the error is still far above the in-distribution level.

Nothing in the training signalled that the model was being used outside its range. The prediction is
delivered with the same confidence either way. **An operator interpolates within its training
distribution and should not be trusted outside it.** A PINN, by contrast, is given the load directly
and does not care where it came from.

This is worth being blunt about, because it is the failure mode most likely to reach a real project.
The middle column is not a degraded answer, it is the wrong answer, and the peak deflection ratio
printed above says by how much. Nothing about the calculation looked different: the same forward
pass, the same milliseconds, the same smooth curve of the same general appearance. A reliability
study or an optimiser driving loads towards the edge of the sampled family would walk into this
without anything in the output changing.

So the training distribution is part of the model and has to be written down and shipped with it:
which modes, what decay, what amplitudes. The statement of validity here is the two lines in Part 1,
$a_n = z_n/n$ with $z_n$ standard normal and $n = 1 \ldots 8$. A surrogate delivered without its
distribution of validity is not a surrogate, it is a number generator.

### Watch the failure arrive

The slider sweeps a single sine mode from $n=1$ upwards. The training set contained modes 1 to 8.
Watch where the prediction stops tracking the exact answer.

In [ ]:
# --- Interactive: walk out of the training distribution ----------------------
mode_err = {}
for n_ in range(1, 21):
    av = np.zeros(n_); av[-1] = 1.0
    c = ood_case(av, f"mode {n_}")
    mode_err[n_] = c

def show_mode(n=1):
    c = mode_err[n]
    fig, (a1x, a2x) = plt.subplots(1, 2, figsize=(11.5, 3.9))
    a1x.plot(x_plot, c["we"], color=C_ALT, lw=3, alpha=0.6, label="exact")
    a1x.plot(x_plot, c["wd"], color=C_DATA, lw=1.5, ls="--", label="DeepONet")
    a1x.plot(x_plot, c["wf"], color=C_FIT,  lw=1.5, ls=":",  label="FNO")
    a1x.invert_yaxis(); a1x.set_xlabel("$x$ (m)"); a1x.set_ylabel("$w$ (m)")
    a1x.set_title(f"$q = \\sin({n}\\pi x/L)$", fontsize=10); a1x.legend(fontsize=8)

    ns = np.arange(1, 21)
    a2x.semilogy(ns, [mode_err[k]["ed"] for k in ns], "o-", color=C_DATA,
                 lw=1.6, ms=4, label="DeepONet")
    a2x.semilogy(ns, [mode_err[k]["ef"] for k in ns], "s-", color=C_FIT,
                 lw=1.6, ms=4, label="FNO")
    a2x.axvspan(0.5, N_MODES + 0.5, color=C_ALT, alpha=0.12)
    a2x.text(4.2, a2x.get_ylim()[1], " trained here", fontsize=8, va="top", color=C_ALT)
    a2x.axvline(n, color="k", ls="--", lw=1)
    a2x.set_xlabel("mode number $n$"); a2x.set_ylabel("relative $L_2$ error")
    a2x.set_title("error against mode number", fontsize=10); a2x.legend(fontsize=8)
    plt.tight_layout(); plt.show()

interact(show_mode, n=IntSlider(1, min=1, max=20, step=1, continuous_update=False,
                                description="mode n"));


---

# Part 5: Discretisation behaviour

### Why the FNO can change grid and the DeepONet branch cannot

The FNO's learned parameters are indexed by Fourier mode, not by grid node. Feed it a finer grid and
the transform simply returns more coefficients; the ones beyond $k_{\max}$ are discarded as they
always were, and the pointwise layers act identically at every point. Nothing in the parameter
tensor has the wrong shape.

The DeepONet is half and half. Its trunk takes a coordinate, so the **output** can be queried
anywhere at any density, which is genuinely useful. Its branch takes a vector of exactly 64 sensor
values, so the **input** resolution is fixed by construction. Changing the sensor layout means a new
branch network.

The claim to test is that the FNO trained on 64 points still works on 128, 256 and 512 without
retraining. The loads here are band-limited by construction, so the transfer should be good, but the
number is worth measuring rather than asserting.

In [ ]:
# --- Evaluate the 64-point FNO on finer grids --------------------------------
RESOLUTIONS = [32, 64, 128, 256, 512]
rng_r = np.random.default_rng(5)
A_res = sample_coeffs(200, rng_r)

print(f"FNO trained on {N_GRID} points, evaluated without retraining.")
print(f"How many Fourier coefficients each grid carries, against the {fno.spec[0].modes} the")
print("network keeps, so we know whether truncation is even in play:")
print(f"{'grid':>6s} {'rfft bins':>11s} {'modes kept':>12s}")
for N_ in RESOLUTIONS:
    print(f"{N_:6d} {N_//2 + 1:>11d} {min(fno.spec[0].modes, N_//2 + 1):>12d}")
print()
print(f"{'grid':>6s} {'FNO rel L2':>14s} {'DeepONet trunk':>16s}")
res_fno, res_don = [], []
for N_ in RESOLUTIONS:
    xr = np.linspace(0, L, N_)
    Qr = q_from_coeffs(A_res, xr); Wr = w_from_coeffs(A_res, xr)
    with torch.no_grad():
        Pf = fno(torch.tensor(Qr / Q_SCALE, dtype=torch.float32)).numpy() * W_SCALE
    ef = rel_l2_rows(Pf, Wr)
    # DeepONet: branch still needs the 64 sensors, only the trunk query grid changes
    Q64 = q_from_coeffs(A_res, x_grid)
    with torch.no_grad():
        Pd = don(torch.tensor(Q64 / Q_SCALE, dtype=torch.float32),
                 torch.tensor(xr.reshape(-1, 1), dtype=torch.float32)).numpy() * W_SCALE
    ed = rel_l2_rows(Pd, Wr)
    res_fno.append(ef); res_don.append(ed)
    tag = "  <- training grid" if N_ == N_GRID else ""
    print(f"{N_:6d} {ef:14.4e} {ed:16.4e}{tag}")


In [ ]:
# --- The same test for a load that is not band-limited -----------------------
print("Now the step load, which has no band limit:")
print(f"{'grid':>6s} {'FNO rel L2':>14s}")
res_step = []
for N_ in RESOLUTIONS:
    xr = np.linspace(0, L, N_)
    qr = q_step_fun(xr).reshape(1, -1)
    wr = w_from_coeffs(a_step, xr)
    with torch.no_grad():
        pr = fno(torch.tensor(qr / Q_SCALE, dtype=torch.float32)).numpy() * W_SCALE
    e = rel_l2_rows(pr, wr)
    res_step.append(e)
    print(f"{N_:6d} {e:14.4e}")

i64 = RESOLUTIONS.index(N_GRID)
print()
print(f"band-limited loads: FNO error at 512 points is "
      f"{res_fno[-1]/res_fno[i64]:.1f}x its error at the {N_GRID}-point training grid")
print(f"step load:          FNO error at 512 points is "
      f"{res_step[-1]/res_step[i64]:.1f}x its error at the {N_GRID}-point training grid")

fig, (a1x, a2x, a3x) = plt.subplots(1, 3, figsize=(14, 4))
a1x.loglog(RESOLUTIONS, res_fno, "o-", color=C_FIT, lw=2, ms=7, label="FNO")
a1x.loglog(RESOLUTIONS, res_don, "s-", color=C_DATA, lw=2, ms=7,
           label="DeepONet (trunk queried, 64 sensors)")
a1x.loglog(RESOLUTIONS, res_step, "^--", color=C_BAD, lw=2, ms=7, label="FNO, step load")
a1x.axvline(N_GRID, color="k", ls=":", lw=1.2)
# headroom so the label sits above every curve instead of inside the legend
a1x.set_ylim(top=max(max(res_step), max(res_fno), max(res_don)) * 2.4)
a1x.text(N_GRID*1.07, a1x.get_ylim()[1]*0.93, "trained here", fontsize=8, va="top")
a1x.set_xlabel("evaluation grid points"); a1x.set_ylabel("mean relative $L_2$ error")
a1x.set_title("error against evaluation resolution", fontsize=10)
a1x.legend(fontsize=7.5, loc="lower right")

xr = np.linspace(0, L, 512)
w512_e = w_from_coeffs(A_res[:1], xr).ravel()
with torch.no_grad():
    p512 = fno(torch.tensor(q_from_coeffs(A_res[:1], xr)/Q_SCALE,
                            dtype=torch.float32)).numpy().ravel() * W_SCALE
a2x.plot(xr, w512_e, color=C_ALT, lw=3, alpha=0.6, label="exact")
a2x.plot(xr, p512, color=C_FIT, lw=1.5, ls="--", label="FNO on 512 points")
a2x.invert_yaxis(); a2x.set_xlabel("$x$ (m)"); a2x.set_ylabel("$w$ (m)")
a2x.set_title("trained on 64 points, evaluated on 512", fontsize=10)
a2x.legend(fontsize=8, loc="lower center")

# where does the extra error live?
w64_e = w_from_coeffs(A_res[:1], x_grid).ravel()
with torch.no_grad():
    p64 = fno(torch.tensor(q_from_coeffs(A_res[:1], x_grid)/Q_SCALE,
                           dtype=torch.float32)).numpy().ravel() * W_SCALE
a3x.semilogy(x_grid, np.abs(p64 - w64_e), color=C_DATA, lw=1.4, label="64 points")
a3x.semilogy(xr, np.abs(p512 - w512_e), color=C_FIT, lw=1.4, label="512 points")
a3x.set_xlabel("$x$ (m)"); a3x.set_ylabel("$|w_{pred} - w_{exact}|$ (m)")
a3x.set_title("pointwise error, same load", fontsize=10); a3x.legend(fontsize=8)
plt.tight_layout(); plt.show()


**What the three panels show.** Left: the mean relative $L_2$ error against the number of
evaluation grid points, on log axes, for the FNO on band-limited loads, for the DeepONet with its 64
sensors and only its trunk queried more finely, and for the FNO on the step load. The dotted vertical
line is the grid the models were trained on. Middle: one load, exact against the FNO evaluated on
512 points. Right: the pointwise absolute error along the beam for that same load, at 64 points and
at 512.

Report what the table says, not what the architecture promises.

The FNO does run on every grid without retraining, and that alone is not something a model with a
dense layer sized to the input could do. But the error is not constant. It is lowest on the grid the
model was trained on, rises by the factor printed above as the grid is refined to 512 points, and
then flattens. It is also worse at 32 points. So resolution transfer here means the error stays
bounded and usable, not that it is unchanged.

The 32-point result is worth not explaining away. The obvious story would be that the coarse grid
cannot carry the modes the network keeps, and the table printed above says that is not what happens:
a 32-point real transform returns 17 coefficients and the network keeps 12, so every learned mode is
still there. Something else is responsible, and this notebook does not measure what. Reporting a
number you have measured without a mechanism you have not is the honest option.

The rise has a structural cause worth knowing. The FFT treats the input as periodic, and this beam
is not: the domain is $[0, L]$ with supports at both ends. The pointwise path has to repair the
mismatch at the ends, and it was fitted at one node spacing. The pointwise error plot shows where
that repair is weakest.

The step load is flat across resolutions and sits at a much higher error throughout. Refining the
grid cannot help it, because what limits it is the distribution mismatch from Part 4 rather than the
discretisation. Separating those two effects is the point of running both tests.

### What not to say about this result

"Resolution invariant" is the phrase usually attached to the FNO, and the numbers above do not
support it in its strong form. What the measurement supports is narrower and still useful: the model
**can be evaluated** on a grid it was never trained on, without reshaping a single weight, and the
error it returns there stays within a small factor of its error on the training grid. Those two
things are different claims and only the second one was measured.

Two practical consequences. If you intend to deploy at a resolution, measure the error at that
resolution rather than quoting the training-grid number. And if the factor printed above matters to
you, training on a mixture of resolutions, or on the finest one you will use, costs nothing extra in
architecture and removes the question.

---

# Closing the series

Six notebooks, one argument.

Notebooks 1 to 4 were about **representation**. Volume fraction alone could not see anisotropy at
all, physically motivated descriptors recovered most of it, and a CNN trained inside a teaching
budget did not beat those descriptors. The input you choose sets the ceiling before any model is
fitted.

Notebook 5 was about **supervision**. With the governing equation in the loss, a network can solve a
boundary value problem with no labelled data at all, at the price of a weighting hyperparameter, no
error bound, and one problem solved per training run.

Notebook 6 was about **amortisation**. Paying a large cost once, over a family of problems, buys a
new solution per forward pass. The bill for that is a training set of solved problems and a model
that is only valid inside the distribution those problems came from.

None of the three replaces a finite element solver on a problem a finite element solver already
handles well. They earn their place when the same problem must be solved many times, when the
geometry or the physics makes meshing hard, or when measurements and equations have to be combined.

---

# What to take away

1. An operator learns a map between functions, $q \mapsto w$, so one trained model covers a family
   of load cases. This is a different object from the input-output regression of Notebooks 1 to 4
   and from the single-problem PINN of Notebook 5.
2. DeepONet factorises the operator into a branch, reading the input function at fixed sensors, and
   a trunk, reading the query coordinate. The trunk learns a basis, the branch learns the
   coefficients in it.
3. The FNO parameterises the operator by learned weights on a truncated set of Fourier modes. Those
   weights are indexed by mode rather than by grid node, which is what allows evaluation on a grid
   the model was never trained on.
4. The speed-up over retraining a PINN per load case is large, and it was measured in Part 4. It is
   not free: the operator needed thousands of solved problems first, and that cost belongs in the
   comparison.
5. Accuracy holds inside the training distribution and degrades sharply outside it. Part 4 measured
   the error on a mode the model had never seen and on a discontinuous load, and the model gave no
   warning that it had left familiar ground.
6. Resolution transfer is real but not free, and the two failure modes are separate. Refining the
   grid away from the training resolution raised the FNO's error on band-limited loads by the factor
   printed in Part 5, and left the step load's error flat, because what limits the step load is the
   training distribution rather than the discretisation. Measure both for your problem rather than
   assuming either.

---

# Exercises

### 1. Change the beam
Set `EI = 5.0` and regenerate the dataset. Before retraining, predict what the normalisation
constants `Q_SCALE` and `W_SCALE` will do. Does the trained operator reach the same relative error,
and why is relative error the right measure to compare across stiffnesses?

### 2. Starve the operator
Retrain the DeepONet on 100, 300 and 1000 training pairs instead of 2000, and plot the test error
against training set size. At what point does the up-front cost of generating solutions stop paying
for itself compared with retraining a PINN per case?

### 3. Truncate harder
Rerun the FNO with `modes = 4` and with `modes = 24`. The training loads contain 8 modes. Explain
the error you get in each case, and say what the right choice would be if the loads contained 30
modes.

### 4. Widen the training distribution
Extend `N_MODES` to 20 and retrain. Does the mode 20 failure in Part 4 disappear? What new failure
appears instead, and what does that say about trying to fix distribution problems by enlarging the
training set?

### 5. The simple method wins
Any load in this notebook can be solved exactly by projecting it onto sine modes and dividing each
coefficient by $(n\pi/L)^4$. That is a few lines of numpy. Time it against the operator forward pass
and compare the accuracy. For this beam, which method would you ship, and what feature of a real
problem would have to change before a neural operator was the better choice?

### 6. Operator or solver
A colleague has a nonlinear beam model with no closed-form solution and a finite element code that
takes 40 minutes per load case. They need 5000 load cases for a reliability study. Using the
break-even numbers printed in Part 4 as a template, work out whether training an operator makes
sense, what they would need to generate first, and what you would tell them about the loads their
reliability study samples from.
